# Evaluación Parcial N°2

**Asignatura:** Ingeniería de Soluciones con IA

**Sección:** 009D  
**Proyecto:** Sistema OMM – Hotel El Almendro  
**Integrantes:** Matías Loyola – Matías Cisternas  
**Docente:** Michael Palomino

## 1. Instalación de dependencias

In [1]:
!pip install -q google-genai pypdf openpyxl langgraph

## 2. Conexión con Google Gemini

In [2]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Responde solamente: conexión correcta"
)

print("API Key cargada correctamente")
print("Modelo: gemini-3.5-flash-lite")
print("Respuesta:", response.text)

API Key cargada correctamente
Modelo: gemini-3.5-flash-lite
Respuesta: conexión correcta


## 3. Importaciones y configuración general

In [3]:
import json
import time
import re
from pathlib import Path
from collections import deque
from typing import TypedDict

import numpy as np
import pandas as pd
from pypdf import PdfReader

from google import genai
from google.genai import types
from google.colab import userdata, files

from langgraph.graph import StateGraph, START, END

MODEL_NAME = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIM = 768
TOP_K = 5

api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

print("Configuración cargada")
print("Modelo de lenguaje:", MODEL_NAME)
print("Modelo de embeddings:", EMBEDDING_MODEL)

Configuración cargada
Modelo de lenguaje: gemini-3.5-flash-lite
Modelo de embeddings: gemini-embedding-2


## 4. Funciones para generación de texto y embeddings

In [4]:
def generar_contenido(prompt, json_mode=False, intentos=4):
    config = types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json" if json_mode else "text/plain"
    )

    ultimo_error = None

    for intento in range(intentos):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=config
            )
        except Exception as e:
            ultimo_error = e
            texto_error = str(e)

            if "429" in texto_error or "503" in texto_error:
                time.sleep(2 ** intento)
                continue

            raise

    raise ultimo_error


def generar_embeddings(textos, tipo="documento", intentos=4):
    if isinstance(textos, str):
        textos = [textos]

    textos_preparados = []

    for texto in textos:
        if tipo == "consulta":
            preparado = f"task: search result | query: {texto}"
        else:
            preparado = f"title: none | text: {texto}"

        textos_preparados.append(preparado)

    contenidos = [
        types.Content(
            parts=[types.Part.from_text(text=texto)]
        )
        for texto in textos_preparados
    ]

    ultimo_error = None

    for intento in range(intentos):
        try:
            resultado = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=contenidos,
                config=types.EmbedContentConfig(
                    output_dimensionality=EMBEDDING_DIM
                )
            )

            vectores = [
                embedding.values
                for embedding in resultado.embeddings
            ]

            return np.array(vectores, dtype=np.float32)

        except Exception as e:
            ultimo_error = e
            texto_error = str(e)

            if "429" in texto_error or "503" in texto_error:
                time.sleep(2 ** intento)
                continue

            raise

    raise ultimo_error

## 5. Embeddings

In [5]:
prueba_embedding = generar_embeddings(
    "proveedores de porcelanatos",
    tipo="consulta"
)

print("Forma del embedding:", prueba_embedding.shape)
print("Prueba de embeddings correcta")

Forma del embedding: (1, 768)
Prueba de embeddings correcta


## 6. Carga de archivos

In [6]:
CARPETA_DOCUMENTOS = Path("/content/documentos_omm")
CARPETA_DOCUMENTOS.mkdir(parents=True, exist_ok=True)

EXTENSIONES_SOPORTADAS = {".pdf", ".xlsx", ".xlsm"}


def obtener_archivos_cargados():
    rutas = [
        ruta
        for ruta in CARPETA_DOCUMENTOS.iterdir()
        if ruta.is_file()
        and ruta.suffix.lower() in EXTENSIONES_SOPORTADAS
    ]

    return sorted(
        rutas,
        key=lambda ruta: ruta.name.lower()
    )


def cargar_archivos():
    archivos_nuevos = files.upload()

    for nombre, contenido in archivos_nuevos.items():
        ruta = CARPETA_DOCUMENTOS / nombre
        ruta.write_bytes(contenido)

    rutas = obtener_archivos_cargados()

    print("Archivos agregados en esta carga:", len(archivos_nuevos))

    if archivos_nuevos:
        for nombre in archivos_nuevos:
            print("-", nombre)

    print()
    print("Total de documentos disponibles:", len(rutas))

    for ruta in rutas:
        print("-", ruta.name)

    return rutas


def vaciar_documentos_cargados():
    for ruta in obtener_archivos_cargados():
        ruta.unlink()

    print("Carpeta de documentos vaciada.")

In [8]:
rutas_archivos = cargar_archivos()

Saving REMODELACION CABAÑA SOL.xlsx to REMODELACION CABAÑA SOL.xlsx
Archivos agregados en esta carga: 1
- REMODELACION CABAÑA SOL.xlsx

Total de documentos disponibles: 2
- PROVEEDORES REMODELACIONES.xlsx
- REMODELACION CABAÑA SOL.xlsx


## 7. Extracción y normalización de información

In [9]:
def limpiar_texto(texto):
    if texto is None:
        return ""

    texto = str(texto)

    lineas = [
        linea.strip()
        for linea in texto.splitlines()
        if linea.strip()
    ]

    return "\n".join(lineas)


def dividir_texto(texto, tamano=1200, solapamiento=200):
    texto = limpiar_texto(texto)

    if not texto:
        return []

    fragmentos = []
    inicio = 0

    while inicio < len(texto):
        fin = min(inicio + tamano, len(texto))
        fragmento = texto[inicio:fin].strip()

        if fragmento:
            fragmentos.append(fragmento)

        if fin == len(texto):
            break

        inicio = max(fin - solapamiento, inicio + 1)

    return fragmentos


def detectar_fila_encabezado_excel(ruta, hoja, max_filas=10):
    vista = pd.read_excel(
        ruta,
        sheet_name=hoja,
        header=None,
        nrows=max_filas,
        dtype=str
    ).fillna("")

    if vista.empty:
        return 0

    mejor_fila = 0
    mejor_puntaje = -1

    for indice, fila in vista.iterrows():
        valores = [
            str(valor).strip()
            for valor in fila.tolist()
            if str(valor).strip()
            and str(valor).strip().lower() != "nan"
        ]

        cantidad_no_vacia = len(valores)
        cantidad_texto = sum(
            any(caracter.isalpha() for caracter in valor)
            for valor in valores
        )

        puntaje = cantidad_no_vacia * 2 + cantidad_texto

        if puntaje > mejor_puntaje:
            mejor_puntaje = puntaje
            mejor_fila = indice

    return int(mejor_fila)


def normalizar_columnas(columnas):
    resultado = []

    for numero, columna in enumerate(columnas, start=1):
        nombre = str(columna).strip()

        if (
            not nombre
            or nombre.lower() == "nan"
            or nombre.lower().startswith("unnamed")
        ):
            nombre = f"Columna {numero}"

        resultado.append(nombre)

    return resultado


def extraer_pdf(ruta):
    lector = PdfReader(str(ruta))
    registros = []

    for numero_pagina, pagina in enumerate(lector.pages, start=1):
        texto = limpiar_texto(pagina.extract_text())

        if not texto:
            continue

        for numero_fragmento, fragmento in enumerate(dividir_texto(texto), start=1):
            registros.append({
                "texto": fragmento,
                "archivo": ruta.name,
                "tipo": "PDF",
                "ubicacion": f"Página {numero_pagina}",
                "fragmento": numero_fragmento
            })

    return registros


def extraer_excel(ruta):
    libro = pd.ExcelFile(ruta)
    registros = []

    for hoja in libro.sheet_names:
        fila_encabezado = detectar_fila_encabezado_excel(ruta, hoja)

        df = pd.read_excel(
            ruta,
            sheet_name=hoja,
            header=fila_encabezado,
            dtype=str
        ).fillna("")

        df.columns = normalizar_columnas(df.columns)

        df = df[
            df.apply(
                lambda fila: any(
                    str(valor).strip()
                    for valor in fila.tolist()
                ),
                axis=1
            )
        ]

        lineas = []

        for indice, fila in df.iterrows():
            valores = []

            for columna, valor in fila.items():
                valor = str(valor).strip()

                if valor and valor.lower() != "nan":
                    valores.append(f"{columna}: {valor}")

            if valores:
                lineas.append(
                    f"Fila {indice + fila_encabezado + 2}: "
                    + " | ".join(valores)
                )

        texto_hoja = "\n".join(lineas)

        for numero_fragmento, fragmento in enumerate(dividir_texto(texto_hoja), start=1):
            registros.append({
                "texto": fragmento,
                "archivo": ruta.name,
                "tipo": "Excel",
                "ubicacion": f"Hoja {hoja}",
                "fragmento": numero_fragmento
            })

    return registros


def procesar_archivos(rutas):
    documentos = []

    for ruta in rutas:
        extension = ruta.suffix.lower()

        if extension == ".pdf":
            documentos.extend(extraer_pdf(ruta))

        elif extension in {".xlsx", ".xlsm"}:
            documentos.extend(extraer_excel(ruta))

        else:
            print("Archivo omitido por formato no soportado:", ruta.name)

    print("Documentos procesados:", len(rutas))
    print("Fragmentos generados:", len(documentos))

    return documentos

In [10]:
documentos = procesar_archivos(rutas_archivos)

if documentos:
    archivos_encontrados = sorted({
        documento["archivo"]
        for documento in documentos
    })

    print()
    print("Fuentes detectadas:")

    for archivo in archivos_encontrados:
        fragmentos_archivo = [
            documento
            for documento in documentos
            if documento["archivo"] == archivo
        ]

        print(f"- {archivo}: {len(fragmentos_archivo)} fragmentos")

    print()
    print("Vista previa por archivo:")

    for archivo in archivos_encontrados:
        primer_fragmento = next(
            documento
            for documento in documentos
            if documento["archivo"] == archivo
        )

        print()
        print("=" * 70)
        print("Archivo:", primer_fragmento["archivo"])
        print("Ubicación:", primer_fragmento["ubicacion"])
        print()
        print(primer_fragmento["texto"][:600])

else:
    print("No se generaron fragmentos.")

Documentos procesados: 2
Fragmentos generados: 16

Fuentes detectadas:
- PROVEEDORES REMODELACIONES.xlsx: 6 fragmentos
- REMODELACION CABAÑA SOL.xlsx: 10 fragmentos

Vista previa por archivo:

Archivo: PROVEEDORES REMODELACIONES.xlsx
Ubicación: Hoja Hoja1

Fila 2: NOMBRE PROVEEDOR: Ducasee 1 | PRODUCTO: Piso Flotante -Piso vinilico- Guardapolvos-Papel Mural-cortinas Roller | CONTACTO: Vendedora Elizabeth almarza | MAIL: elizabeth.almarza@dapducasse.cl | TIENDA: Av. Nueva Costanera 3573, Vitacura. | COMENTARIO: LE compramos directo se ha comprado en diversas oportunidades pisos, guardapolvos- somos clientes y tenemos descuento preferente.
Fila 3: NOMBRE PROVEEDOR: Ducasee 2 | PRODUCTO: Griferia-Porta toallas | CONTACTO: vendedor en meson | COMENTARIO: Se le a comprado varias veces porta toallas
Fila 4: NOMBRE PROVEEDOR: Ducasee 3 | PRODUCTO: Porc

Archivo: REMODELACION CABAÑA SOL.xlsx
Ubicación: Hoja Remo-cabaña sol

Fila 3: CATEGORIA: PISOS | PRO0VEDOR: BAUQ | TIPO DE PRODUCTO: 40 mt2 

## 8. Índice de recuperación semántica

In [11]:
class IndiceSemantico:

    def __init__(self):
        self.documentos = []
        self.vectores = np.empty(
            (0, EMBEDDING_DIM),
            dtype=np.float32
        )

    @staticmethod
    def normalizar(vectores):
        normas = np.linalg.norm(
            vectores,
            axis=1,
            keepdims=True
        )

        normas[normas == 0] = 1
        return vectores / normas

    def construir(self, documentos, batch_size=20):
        self.documentos = list(documentos)

        if not documentos:
            print("No hay documentos para indexar.")
            return

        todos_vectores = []

        for inicio in range(0, len(documentos), batch_size):
            lote = documentos[inicio:inicio + batch_size]

            textos = [
                documento["texto"]
                for documento in lote
            ]

            vectores = generar_embeddings(
                textos,
                tipo="documento"
            )

            todos_vectores.append(vectores)

            procesados = min(
                inicio + batch_size,
                len(documentos)
            )

            print(
                f"Embeddings procesados: "
                f"{procesados}/{len(documentos)}"
            )

        matriz = np.vstack(todos_vectores)
        self.vectores = self.normalizar(matriz)

        print("Índice semántico construido.")

    def buscar(self, consulta, top_k=TOP_K):
        if len(self.documentos) == 0:
            return []

        vector_consulta = generar_embeddings(
            consulta,
            tipo="consulta"
        )

        vector_consulta = self.normalizar(
            vector_consulta
        )[0]

        similitudes = self.vectores @ vector_consulta
        cantidad = min(top_k, len(self.documentos))
        indices = np.argsort(similitudes)[::-1][:cantidad]

        resultados = []

        for indice in indices:
            documento = dict(self.documentos[indice])
            documento["similitud"] = float(similitudes[indice])
            resultados.append(documento)

        return resultados

    def listar_fuentes(self):
        return sorted({
            (
                documento["archivo"],
                documento["tipo"],
                documento["ubicacion"]
            )
            for documento in self.documentos
        })


indice_semantico = IndiceSemantico()

print("Índice semántico inicializado.")

Índice semántico inicializado.


In [12]:
indice_semantico.construir(documentos)


def actualizar_base_documental():
    global rutas_archivos
    global documentos

    rutas_archivos = obtener_archivos_cargados()
    documentos = procesar_archivos(rutas_archivos)
    indice_semantico.construir(documentos)

    print()
    print("Base documental actualizada.")
    print("Archivos incluidos:", len(rutas_archivos))

    for ruta in rutas_archivos:
        print("-", ruta.name)

    return documentos

Embeddings procesados: 16/16
Índice semántico construido.


## 9. Prueba de búsqueda semántica

In [13]:
consultas_prueba = [
    "proveedores que venden porcelanatos",
    "productos de la remodelación Cabaña Sol comprados en Easy"
]

for consulta in consultas_prueba:
    print()
    print("=" * 70)
    print("Consulta:", consulta)

    resultados_prueba = indice_semantico.buscar(
        consulta,
        top_k=3
    )

    for numero, resultado in enumerate(resultados_prueba, start=1):
        print()
        print(f"Resultado {numero}")
        print("Archivo:", resultado["archivo"])
        print("Ubicación:", resultado["ubicacion"])
        print("Similitud:", round(resultado["similitud"], 3))
        print(resultado["texto"][:400])


Consulta: proveedores que venden porcelanatos

Resultado 1
Archivo: PROVEEDORES REMODELACIONES.xlsx
Ubicación: Hoja Hoja1
Similitud: 0.741
Fila 2: NOMBRE PROVEEDOR: Ducasee 1 | PRODUCTO: Piso Flotante -Piso vinilico- Guardapolvos-Papel Mural-cortinas Roller | CONTACTO: Vendedora Elizabeth almarza | MAIL: elizabeth.almarza@dapducasse.cl | TIENDA: Av. Nueva Costanera 3573, Vitacura. | COMENTARIO: LE compramos directo se ha comprado en diversas oportunidades pisos, guardapolvos- somos clientes y tenemos descuento preferente.
Fila 3: NOM

Resultado 2
Archivo: PROVEEDORES REMODELACIONES.xlsx
Ubicación: Hoja Hoja1
Similitud: 0.73
y malos
Fila 11: NOMBRE PROVEEDOR: Easy | PRODUCTO: Porcelanatos- ceramicas- pegamento de ceramica y porcelanato-pintura de calidad-wc -marca vesanti(son ergonomicos) | CONTACTO: vendedor area empresa | TIENDA: Av. Portales 3698, San Bernardo, | WEB: easy.cl | COMENTARIO: Se compra continuamente, somos clientes preferentes con descuento
Fila 12: NOMBRE PROVEEDOR: b

## 10. Memoria del agente

In [14]:
ARCHIVO_MEMORIA = Path("/content/memoria_omm.json")


class MemoriaOMM:

    def __init__(self, max_turnos=6):
        self.corto_plazo = deque(maxlen=max_turnos)
        self.largo_plazo = self._cargar_largo_plazo()

    def _cargar_largo_plazo(self):
        if ARCHIVO_MEMORIA.exists():
            try:
                return json.loads(
                    ARCHIVO_MEMORIA.read_text(
                        encoding="utf-8"
                    )
                )
            except Exception:
                return {}

        return {}

    def _guardar_largo_plazo(self):
        ARCHIVO_MEMORIA.write_text(
            json.dumps(
                self.largo_plazo,
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

    def agregar_turno(self, usuario, asistente):
        self.corto_plazo.append({
            "usuario": usuario,
            "asistente": asistente
        })

    def recordar(self, clave, valor):
        self.largo_plazo[clave] = valor
        self._guardar_largo_plazo()

    def limpiar(self):
        self.corto_plazo.clear()
        self.largo_plazo = {}
        self._guardar_largo_plazo()

    def contexto(self):
        partes = []

        if self.largo_plazo:
            partes.append(
                "Memoria persistente:\n" +
                json.dumps(
                    self.largo_plazo,
                    ensure_ascii=False,
                    indent=2
                )
            )

        if self.corto_plazo:
            historial = []

            for turno in self.corto_plazo:
                historial.append(
                    "Usuario: "
                    + turno["usuario"]
                    + "\nAsistente: "
                    + turno["asistente"]
                )

            partes.append(
                "Conversación reciente:\n"
                + "\n\n".join(historial)
            )

        return "\n\n".join(partes) if partes else "Sin memoria previa."


memoria = MemoriaOMM()

print("Memoria inicializada.")

Memoria inicializada.


## 11. Herramientas del agente

In [15]:
CARPETA_REPORTES = Path("/content/reportes_omm")
CARPETA_REPORTES.mkdir(parents=True, exist_ok=True)


def herramienta_buscar_documentos(consulta, top_k=TOP_K):
    resultados = indice_semantico.buscar(
        consulta,
        top_k=top_k
    )

    if not resultados:
        return {
            "contexto": "No hay documentos indexados o no se encontraron resultados.",
            "fuentes": []
        }

    bloques = []
    fuentes = []

    for numero, resultado in enumerate(resultados, start=1):
        fuente = (
            f"{resultado['archivo']} | "
            f"{resultado['ubicacion']} | "
            f"similitud={resultado['similitud']:.3f}"
        )

        bloques.append(
            f"[Fuente {numero}: {fuente}]\n"
            f"{resultado['texto']}"
        )

        fuentes.append(fuente)

    return {
        "contexto": "\n\n".join(bloques),
        "fuentes": fuentes
    }


def herramienta_listar_documentos():
    fuentes = indice_semantico.listar_fuentes()

    if not fuentes:
        return "No hay documentos indexados."

    return "\n".join(
        f"- {archivo} | {tipo} | {ubicacion}"
        for archivo, tipo, ubicacion in fuentes
    )


def herramienta_analizar_documentos(consulta):
    recuperado = herramienta_buscar_documentos(
        consulta,
        top_k=TOP_K
    )

    prompt = f'''
Analiza exclusivamente la información recuperada desde los documentos OMM.

Consulta:
{consulta}

Contexto:
{recuperado["contexto"]}

Instrucciones:
- Compara los registros cuando corresponda.
- Explica brevemente el criterio utilizado.
- No inventes información que no aparezca en el contexto.
- Si la evidencia no permite una conclusión, indícalo.
'''

    analisis = generar_contenido(prompt).text.strip()

    return {
        "analisis": analisis,
        "contexto": recuperado["contexto"],
        "fuentes": recuperado["fuentes"]
    }


def herramienta_guardar_memoria(clave, valor):
    memoria.recordar(clave, valor)

    return (
        f"Información guardada correctamente. "
        f"Clave: '{clave}'. Valor: '{valor}'."
    )


def nombre_archivo_seguro(titulo):
    titulo = titulo.strip().lower()
    titulo = re.sub(r"[^a-záéíóúñ0-9]+", "_", titulo)
    titulo = titulo.strip("_")

    return titulo[:60] or "reporte_omm"


def herramienta_escribir_reporte(titulo, contenido):
    nombre = nombre_archivo_seguro(titulo) + ".txt"
    ruta = CARPETA_REPORTES / nombre

    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

    return str(ruta)

## 12. Planificación y toma de decisiones

In [16]:
HERRAMIENTAS_VALIDAS = {
    "buscar_documentos",
    "listar_documentos",
    "analizar_documentos",
    "guardar_memoria",
    "generar_reporte",
    "respuesta_general"
}


def planificar_consulta(pregunta):

    prompt = f'''
Eres el planificador del agente OMM del Hotel El Almendro.

Debes seleccionar UNA herramienta.

Herramientas:

1. buscar_documentos
Para recuperar datos concretos o responder preguntas factuales sobre PDF o Excel.

2. listar_documentos
Para indicar qué documentos, hojas o fuentes están disponibles.

3. analizar_documentos
Para comparar, evaluar, priorizar, identificar el mejor o peor registro, ordenar por criterio o razonar sobre varios datos del documento.

4. guardar_memoria
Cuando el usuario diga explícitamente que recuerdes, guardes o conserves un dato para consultas posteriores.

5. generar_reporte
Cuando el usuario pida crear, redactar, generar o guardar un reporte, informe, registro o resumen escrito basado en los documentos.

6. respuesta_general
Para preguntas que puedan contestarse con la memoria ya almacenada o que no requieran consultar documentos.

Reglas importantes:
- Si la pregunta solicita recordar un dato, usa guardar_memoria.
- Si pregunta por un dato que el usuario ya pidió recordar y aparece en la memoria, usa respuesta_general.
- Si pide "peor", "mejor", "compara", "analiza", "prioriza" u otra evaluación sobre documentos, usa analizar_documentos.
- Si pide crear o guardar un reporte/informe, usa generar_reporte.
- No inventes herramientas.

Memoria disponible:
{memoria.contexto()}

Consulta:
{pregunta}

Devuelve únicamente JSON con esta estructura:

{{
  "herramienta": "buscar_documentos | listar_documentos | analizar_documentos | guardar_memoria | generar_reporte | respuesta_general",
  "consulta_busqueda": "",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": ""
}}
'''

    respuesta = generar_contenido(
        prompt,
        json_mode=True
    )

    try:
        plan = json.loads(respuesta.text)
    except Exception:
        plan = {
            "herramienta": "buscar_documentos",
            "consulta_busqueda": pregunta,
            "clave_memoria": "",
            "valor_memoria": "",
            "titulo_reporte": "",
            "objetivo": "Recuperar información relacionada con la consulta."
        }

    herramienta = plan.get(
        "herramienta",
        "buscar_documentos"
    )

    if herramienta not in HERRAMIENTAS_VALIDAS:
        herramienta = "buscar_documentos"

    plan["herramienta"] = herramienta

    if not plan.get("consulta_busqueda"):
        plan["consulta_busqueda"] = pregunta

    return plan

## 13. Orquestación con LangGraph

In [17]:
class EstadoAgente(TypedDict, total=False):
    pregunta: str
    plan: dict
    contexto: str
    fuentes: list
    respuesta: str
    archivo_generado: str


def nodo_planificador(estado):
    return {
        "plan": planificar_consulta(
            estado["pregunta"]
        )
    }


def ruta_planificador(estado):
    return estado["plan"]["herramienta"]


def nodo_buscar_documentos(estado):
    consulta = estado["plan"].get(
        "consulta_busqueda",
        estado["pregunta"]
    )

    resultado = herramienta_buscar_documentos(
        consulta
    )

    return {
        "contexto": resultado["contexto"],
        "fuentes": resultado["fuentes"]
    }


def nodo_listar_documentos(estado):
    return {
        "contexto": herramienta_listar_documentos(),
        "fuentes": []
    }


def nodo_analizar_documentos(estado):
    consulta = estado["plan"].get(
        "consulta_busqueda",
        estado["pregunta"]
    )

    resultado = herramienta_analizar_documentos(
        consulta
    )

    return {
        "contexto": (
            resultado["contexto"]
            + "\n\nAnálisis realizado:\n"
            + resultado["analisis"]
        ),
        "fuentes": resultado["fuentes"]
    }


def nodo_guardar_memoria(estado):
    plan = estado["plan"]

    clave = (
        plan.get("clave_memoria")
        or "dato_usuario"
    )

    valor = (
        plan.get("valor_memoria")
        or estado["pregunta"]
    )

    confirmacion = herramienta_guardar_memoria(
        clave,
        valor
    )

    return {
        "contexto": confirmacion,
        "fuentes": [],
        "respuesta": confirmacion
    }


def nodo_generar_reporte(estado):
    plan = estado["plan"]

    consulta = plan.get(
        "consulta_busqueda",
        estado["pregunta"]
    )

    titulo = (
        plan.get("titulo_reporte")
        or "Reporte OMM"
    )

    recuperado = herramienta_buscar_documentos(
        consulta,
        top_k=TOP_K
    )

    prompt = f'''
Genera un reporte operacional breve para el Sistema OMM del Hotel El Almendro.

Título:
{titulo}

Solicitud:
{estado["pregunta"]}

Información recuperada:
{recuperado["contexto"]}

Reglas:
- Usa solamente la información recuperada.
- No inventes datos.
- Estructura el reporte con título, resumen y hallazgos.
- Mantén un tono técnico y claro.
'''

    contenido = generar_contenido(
        prompt
    ).text.strip()

    ruta = herramienta_escribir_reporte(
        titulo,
        contenido
    )

    respuesta = (
        f"Reporte generado y guardado correctamente en:\n{ruta}\n\n"
        f"{contenido}"
    )

    return {
        "contexto": contenido,
        "fuentes": recuperado["fuentes"],
        "respuesta": respuesta,
        "archivo_generado": ruta
    }


def nodo_respuesta_general(estado):
    return {
        "contexto": "No fue necesario consultar documentos para esta pregunta.",
        "fuentes": []
    }


def nodo_responder(estado):
    prompt = f'''
Eres el agente OMM del Hotel El Almendro.

Consulta:
{estado["pregunta"]}

Plan:
{json.dumps(
    estado.get("plan", {}),
    ensure_ascii=False
)}

Memoria:
{memoria.contexto()}

Contexto disponible:
{estado.get("contexto", "")}

Reglas:
- Responde de forma clara y breve.
- Para información documental, usa solamente el contexto recuperado.
- Cuando corresponda, identifica el archivo y la hoja de donde proviene la información.
- Para preguntas sobre datos que el usuario pidió recordar, usa la memoria.
- No inventes información.
- Si falta evidencia, indícalo.
'''

    respuesta = generar_contenido(
        prompt
    ).text.strip()

    fuentes = estado.get(
        "fuentes",
        []
    )

    if fuentes:
        fuentes_unicas = list(
            dict.fromkeys(fuentes)
        )

        respuesta += (
            "\n\nFuentes consultadas:\n"
            + "\n".join(
                f"- {fuente}"
                for fuente in fuentes_unicas
            )
        )

    return {
        "respuesta": respuesta
    }


def nodo_memoria(estado):
    memoria.agregar_turno(
        estado["pregunta"],
        estado["respuesta"]
    )

    return {}


grafo = StateGraph(EstadoAgente)

grafo.add_node("planificador", nodo_planificador)
grafo.add_node("buscar_documentos", nodo_buscar_documentos)
grafo.add_node("listar_documentos", nodo_listar_documentos)
grafo.add_node("analizar_documentos", nodo_analizar_documentos)
grafo.add_node("guardar_memoria", nodo_guardar_memoria)
grafo.add_node("generar_reporte", nodo_generar_reporte)
grafo.add_node("respuesta_general", nodo_respuesta_general)
grafo.add_node("responder", nodo_responder)
grafo.add_node("memoria", nodo_memoria)

grafo.add_edge(START, "planificador")

grafo.add_conditional_edges(
    "planificador",
    ruta_planificador,
    {
        "buscar_documentos": "buscar_documentos",
        "listar_documentos": "listar_documentos",
        "analizar_documentos": "analizar_documentos",
        "guardar_memoria": "guardar_memoria",
        "generar_reporte": "generar_reporte",
        "respuesta_general": "respuesta_general"
    }
)

grafo.add_edge("buscar_documentos", "responder")
grafo.add_edge("listar_documentos", "responder")
grafo.add_edge("analizar_documentos", "responder")
grafo.add_edge("respuesta_general", "responder")

grafo.add_edge("guardar_memoria", "memoria")
grafo.add_edge("generar_reporte", "memoria")

grafo.add_edge("responder", "memoria")
grafo.add_edge("memoria", END)

agente_omm = grafo.compile()

print("Agente OMM compilado correctamente.")

Agente OMM compilado correctamente.


## 14. Función principal del agente

In [18]:
def preguntar_omm(pregunta, mostrar_plan=True):
    resultado = agente_omm.invoke({
        "pregunta": pregunta
    })

    if mostrar_plan:
        print("Plan seleccionado:")
        print(
            json.dumps(
                resultado.get("plan", {}),
                ensure_ascii=False,
                indent=2
            )
        )
        print()

    print("Respuesta:")
    print(resultado["respuesta"])

    if resultado.get("archivo_generado"):
        print()
        print(
            "Archivo generado:",
            resultado["archivo_generado"]
        )

    return resultado

## 15. Prueba inicial del agente

In [19]:
resultado_inicial = preguntar_omm(
    "¿Qué productos de la remodelación Cabaña Sol fueron comprados en Easy?"
)

Plan seleccionado:
{
  "herramienta": "buscar_documentos",
  "consulta_busqueda": "productos remodelacion Cabana Sol comprados en Easy",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Recuperar los productos de la remodelación de la Cabaña Sol que fueron comprados en Easy a partir de los documentos disponibles."
}

Respuesta:
Según los documentos del Hotel El Almendro (archivo `REMODELACION CABAÑA SOL.xlsx`, hoja `Remo-cabaña sol`), los productos comprados en **EASY** para la remodelación de la Cabaña Sol son los siguientes:

1. **Simplísima Madera Veteada Soft 6mm** (33 Mt2 o Uni)
2. **Simplísima Madera Veteada Soft 6mm** (9 Mt2 o Uni)
3. **Adhesivo de montaje en oferta 3x2 Agorex PL 500** (para pegar placas Simplísimas al muro, 80 Mt2 o Uni)
4. **Adhesivo de montaje en oferta 3x2 Agorex PL 500** (para pegar guardapolvos y junquillos, 12 Mt2 o Uni)
5. **Bisagra clip recta** (10 Mt2 o Uni)
6. **Manilla Bruselas** (4 Mt2 o Uni)
7. **Corredera telesc

## 16. Chat interactivo

In [20]:
def iniciar_chat():

    print("Agente OMM iniciado.")
    print("Escribe 'salir' para terminar.\n")

    while True:
        pregunta = input("Usuario: ").strip()

        if pregunta.lower() in {
            "salir",
            "exit",
            "terminar"
        }:
            print("Conversación finalizada.")
            break

        if not pregunta:
            continue

        resultado = agente_omm.invoke({
            "pregunta": pregunta
        })

        print("\nHerramienta seleccionada:")
        print(
            resultado.get(
                "plan",
                {}
            ).get(
                "herramienta",
                "No identificada"
            )
        )

        print("\nAgente OMM:")
        print(resultado["respuesta"])

        if resultado.get("archivo_generado"):
            print(
                "\nArchivo generado:",
                resultado["archivo_generado"]
            )

        print()

In [21]:
iniciar_chat()

Agente OMM iniciado.
Escribe 'salir' para terminar.

Usuario: que documentos tienes disponibles 

Herramienta seleccionada:
listar_documentos

Agente OMM:
Como agente OMM del Hotel El Almendro, tengo disponibles los siguientes documentos en mi base de datos:

1. **PROVEEDORES REMODELACIONES.xlsx** (Hoja: *Hoja1*)
2. **REMODELACION CABAÑA SOL.xlsx** (Hoja: *Remo-cabaña sol*)

Usuario: cual es el proveedor que peor comentarios tiene y porque motivo 

Herramienta seleccionada:
analizar_documentos

Agente OMM:
Según los documentos del Hotel El Almendro (archivo `PROVEEDORES REMODELACIONES.xlsx`, hoja `Hoja1`), el proveedor con peores comentarios es **Roca** (referenciado también como `chc.cl`).

**Motivo de los comentarios negativos:**
1. El lugar de retiro en Lampa atiende muy mal.
2. Demoraron 2 horas en entregar los productos.
3. Los WCs adquiridos no gustaron, ya que se indica que no son ergonómicos, además de ser caros y malos.

Fuentes consultadas:
- PROVEEDORES REMODELACIONES.xlsx |

## 17. Casos de prueba

In [22]:
memoria.limpiar()

for archivo in CARPETA_REPORTES.glob("*.txt"):
    archivo.unlink()

print("Entorno preparado para los casos de prueba.")
print("Documentos disponibles:", len(rutas_archivos))

for ruta in rutas_archivos:
    print("-", ruta.name)

print("Datos persistentes:", len(memoria.largo_plazo))
print("Turnos de corto plazo:", len(memoria.corto_plazo))

Entorno preparado para los casos de prueba.
Documentos disponibles: 2
- PROVEEDORES REMODELACIONES.xlsx
- REMODELACION CABAÑA SOL.xlsx
Datos persistentes: 0
Turnos de corto plazo: 0


In [23]:
caso_1 = preguntar_omm(
    "¿Qué proveedores venden porcelanatos?"
)

Plan seleccionado:
{
  "herramienta": "buscar_documentos",
  "consulta_busqueda": "proveedores que venden porcelanatos",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Buscar en los documentos disponibles qué proveedores ofrecen porcelanatos."
}

Respuesta:
Según el archivo **PROVEEDORES REMODELACIONES.xlsx** (Hoja `Hoja1`), los proveedores que venden porcelanatos son:

1. **Ducasee 3** (Producto: Porcelanatos - Cerámicas)
2. **Budnik** (Producto: Grifería, duchas, lavamanos flotantes, piso flotante, piso porcelanato. Tienda: Av. Presidente Kennedy 7600 | Web: www.budnik.cl)
3. **Atika** (Producto: Porcelanatos, pisos, lavamanos, grifería. Tienda: Av. Vitacura 5770 | Web: www.atika.cl)
4. **Easy** (Producto: Porcelanatos, cerámicas, pegamento de cerámica y porcelanato, pintura de calidad, WC. Tienda: Av. Portales 3698, San Bernardo | Web: easy.cl)

Fuentes consultadas:
- PROVEEDORES REMODELACIONES.xlsx | Hoja Hoja1 | similitud=0.741
- PROVEEDORES R

In [24]:
caso_2 = preguntar_omm(
    "¿Qué productos de la remodelación Cabaña Sol fueron comprados en Easy?"
)

Plan seleccionado:
{
  "herramienta": "buscar_documentos",
  "consulta_busqueda": "productos de la remodelación Cabaña Sol comprados en Easy",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Buscar en los documentos qué productos de la remodelación de la Cabaña Sol fueron adquiridos en el proveedor Easy."
}

Respuesta:
Según el archivo **REMODELACION CABAÑA SOL.xlsx** (Hoja `Remo-cabaña sol`), los productos comprados en **Easy** para la remodelación de la Cabaña Sol son:

1. **Simplísima madera veteada soft 6mm** (33 m² y 9 m²)
2. **Adhesivo de montaje en oferta 3x2 Agorex PL 500** (para pegar placas Simplísimas al muro y para pegar guardapolvos y junquillos)
3. **Bisagra clip recta** (10 unidades)
4. **Manilla Bruselas** (4 unidades)
5. **Corredera telescópica** (1 unidad)
6. **LZ FCB4WBL A** (2 unidades)
7. **Spray anticorrosivo blanco** (2 unidades)
8. **Ampolleta LED 7W luz cálida** (8 unidades)
9. **Mono mando lavaplatos** (1 unidad)
10. **Apli

In [25]:
caso_3 = preguntar_omm(
    "¿Qué documentos y hojas tienes disponibles para consultar?"
)

Plan seleccionado:
{
  "herramienta": "listar_documentos",
  "consulta_busqueda": "¿Qué documentos y hojas tienes disponibles para consultar?",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Listar los documentos, hojas o fuentes disponibles para consultar."
}

Respuesta:
Como agente OMM del Hotel El Almendro, tengo disponibles los siguientes documentos y hojas para consultar:

1. **PROVEEDORES REMODELACIONES.xlsx** (Hoja: `Hoja1`)
2. **REMODELACION CABAÑA SOL.xlsx** (Hoja: `Remo-cabaña sol`)


In [26]:
caso_4 = preguntar_omm(
    "Analiza los comentarios de los proveedores e identifica cuál presenta el comentario más negativo. Explica brevemente tu elección."
)

Plan seleccionado:
{
  "herramienta": "analizar_documentos",
  "consulta_busqueda": "comentarios de los proveedores",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Analizar los comentarios de los proveedores para identificar cuál presenta el comentario más negativo y explicar la elección."
}

Respuesta:
Como agente OMM del Hotel El Almendro, basándome en el archivo **PROVEEDORES REMODELACIONES.xlsx** (Hoja `Hoja1`), el proveedor que presenta el **comentario más negativo** es **Roca**.

**Explicación de la elección:**
El comentario registrado para Roca detalla múltiples problemas graves tanto en el servicio logístico como en la calidad del producto:
1. **Mala atención y demoras:** El retiro de los productos se debe realizar en Lampa, un lugar que "atiende muy mal" y donde demoraron dos horas en realizar la entrega.
2. **Insatisfacción con el producto:** Se señala directamente que los WC no gustaron, calificándolos explícitamente como "no ergonómico

In [27]:
caso_5 = preguntar_omm(
    "Recuerda que para esta prueba el proveedor prioritario es Atika."
)

Plan seleccionado:
{
  "herramienta": "guardar_memoria",
  "consulta_busqueda": "Recuerda que para esta prueba el proveedor prioritario es Atika.",
  "clave_memoria": "proveedor_prioritario",
  "valor_memoria": "Atika",
  "titulo_reporte": "",
  "objetivo": "Guardar en la memoria que el proveedor prioritario para la prueba es Atika."
}

Respuesta:
Información guardada correctamente. Clave: 'proveedor_prioritario'. Valor: 'Atika'.


In [28]:
caso_6 = preguntar_omm(
    "¿Cuál es el proveedor prioritario que te indiqué?"
)

Plan seleccionado:
{
  "herramienta": "respuesta_general",
  "consulta_busqueda": "¿Cuál es el proveedor prioritario que te indiqué?",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "",
  "objetivo": "Responder a la pregunta del usuario utilizando la información almacenada en la memoria persistente sobre el proveedor prioritario."
}

Respuesta:
Como agente OMM del Hotel El Almendro, te indico que el proveedor prioritario registrado es **Atika**.


In [29]:
caso_7 = preguntar_omm(
    "Genera y guarda un reporte breve sobre los proveedores que venden porcelanatos y sus comentarios."
)

Plan seleccionado:
{
  "herramienta": "generar_reporte",
  "consulta_busqueda": "proveedores que venden porcelanatos y sus comentarios",
  "clave_memoria": "",
  "valor_memoria": "",
  "titulo_reporte": "Reporte de Proveedores de Porcelanatos",
  "objetivo": "Crear y guardar un reporte escrito basado en los documentos sobre los proveedores que venden porcelanatos y sus respectivos comentarios."
}

Respuesta:
Reporte generado y guardado correctamente en:
/content/reportes_omm/reporte_de_proveedores_de_porcelanatos.txt

# Sistema OMM - Hotel El Almendro

## Reporte de Proveedores de Porcelanatos

### Resumen
El presente reporte operacional detalla los proveedores identificados en el registro de remodelaciones que ofrecen productos de porcelanatos, incluyendo su información de contacto, ubicación y los comentarios asociados a cada uno dentro de la gestión de compras del Hotel El Almendro.

---

### Hallazgos

A partir de la información recuperada, se identificaron los siguientes proveedor

## 18. Validación de decisiones

In [30]:
validaciones = [
    ("Caso 1", caso_1, "buscar_documentos"),
    ("Caso 2", caso_2, "buscar_documentos"),
    ("Caso 3", caso_3, "listar_documentos"),
    ("Caso 4", caso_4, "analizar_documentos"),
    ("Caso 5", caso_5, "guardar_memoria"),
    ("Caso 6", caso_6, "respuesta_general"),
    ("Caso 7", caso_7, "generar_reporte")
]

for nombre, resultado, esperada in validaciones:
    obtenida = resultado.get("plan", {}).get("herramienta", "No identificada")
    estado = "CORRECTO" if obtenida == esperada else "REVISAR"

    print(
        f"{nombre}: "
        f"esperada={esperada} | "
        f"obtenida={obtenida} | "
        f"{estado}"
    )

Caso 1: esperada=buscar_documentos | obtenida=buscar_documentos | CORRECTO
Caso 2: esperada=buscar_documentos | obtenida=buscar_documentos | CORRECTO
Caso 3: esperada=listar_documentos | obtenida=listar_documentos | CORRECTO
Caso 4: esperada=analizar_documentos | obtenida=analizar_documentos | CORRECTO
Caso 5: esperada=guardar_memoria | obtenida=guardar_memoria | CORRECTO
Caso 6: esperada=respuesta_general | obtenida=respuesta_general | CORRECTO
Caso 7: esperada=generar_reporte | obtenida=generar_reporte | CORRECTO


## 19. Verificación final del sistema

In [31]:
reportes_generados = list(
    CARPETA_REPORTES.glob("*.txt")
)

print("Modelo de lenguaje:", MODEL_NAME)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("Archivos cargados:", len(rutas_archivos))

for ruta in rutas_archivos:
    print("-", ruta.name)

print("Fragmentos indexados:", len(indice_semantico.documentos))
print("Fuentes disponibles:", len(indice_semantico.listar_fuentes()))
print("Turnos en memoria de corto plazo:", len(memoria.corto_plazo))
print("Datos en memoria persistente:", len(memoria.largo_plazo))
print("Reportes escritos:", len(reportes_generados))

for reporte in reportes_generados:
    print("-", reporte)

print("Agente compilado:", agente_omm is not None)

Modelo de lenguaje: gemini-3.5-flash-lite
Modelo de embeddings: gemini-embedding-2
Archivos cargados: 2
- PROVEEDORES REMODELACIONES.xlsx
- REMODELACION CABAÑA SOL.xlsx
Fragmentos indexados: 16
Fuentes disponibles: 2
Turnos en memoria de corto plazo: 6
Datos en memoria persistente: 1
Reportes escritos: 1
- /content/reportes_omm/reporte_de_proveedores_de_porcelanatos.txt
Agente compilado: True


## Reflexiones individuales

### Matías Loyola
Logré entender mejor cómo se procesa la información de los archivos, ya que primero se cargan y extraen sus datos, luego se generan embeddings para buscar el contexto más relacionado con la pregunta, y finalmente ese contexto se entrega a Gemini para generar una respuesta basada en los documentos.

### Matías Cisternas
Cómo el agente utiliza distintas herramientas según la consulta y cómo la memoria permite mantener información entre preguntas. También entendí mejor la función del planificador para decidir qué acción ejecutar. Mi aporte estuvo principalmente en probar los distintos casos y revisar que el flujo y las respuestas funcionaran correctamente

## Referencias técnicas

- Google. (2024). *Google GenAI SDK Documentation*. Google Cloud. https://ai.google.dev/gemini-api/docs
- LangChain Inc. (2024). *LangGraph: Build resilient language agents as graphs*. https://langchain-ai.github.io/langgraph/
- Palomino, M. (2024). *Ingeniería de Soluciones con Inteligencia Artificial: https://github.com/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial
- OpenAI. (2024). ChatGPT (Versión GPT-4o) [Modelo de lenguaje de gran escala]. https://chatgpt.com/